# Proactive Churn — Redshift connection & dataset extract

Connects to the brightwheel Analytics Redshift cluster and builds the v1 modelling dataset.

**Before you run anything:**

1. **Be on VPN.** The cluster is not reachable off-VPN — you get a connection timeout, not an auth error. ([VPN setup guide](https://docs.google.com/document/d/1kNqdf4H1voZC0ne0UI1oH-K66tn8EBZ0OfC-Ivh-wUc/edit))
2. **Fill in `.env`** at the project root (`cp .env.example .env`). Credentials live there, never in this notebook.
3. **Kernel = "Python (Proactive Churn)"** — created by `setup.sh`.

Credentials were issued by Shana Luo (Data Eng) on the `bw_analytics` database. Password rotation steps are in `README.md`.

## 1. Imports

Connection and extract only. Modeling libraries are deliberately absent — see the note in the cell.

In [ ]:
# --- stdlib ---
import os
import time
from contextlib import contextmanager
from pathlib import Path

# --- data ---
import pandas as pd

# --- redshift ---
import redshift_connector
from dotenv import load_dotenv

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

print(f"pandas             {pd.__version__}")
print(f"redshift_connector {redshift_connector.__version__}")

# xgboost / shap / sklearn / BorutaShap are NOT imported here — this notebook is
# connection + extract only, and importing them costs several seconds per kernel
# start. setup.sh verifies the full stack. If you add modelling cells here, note
# that BorutaShap needs the scipy.stats.binom_test shim documented in README.md.

## 2. Credentials and paths

Loaded from the gitignored `.env` at the project root. Nothing secret is printed or stored in this notebook.

`data/raw/` is gitignored — school-level customer data must not land in git. An extract pulled with your credentials is no more shareable than `analytics_mart_customer` itself is.

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "data" else Path.cwd()
DOTENV_PATH = PROJECT_ROOT / ".env"

if not DOTENV_PATH.exists():
    raise FileNotFoundError(
        f"No .env at {DOTENV_PATH}. Run: cp .env.example .env, then fill in REDSHIFT_PASSWORD"
    )

load_dotenv(DOTENV_PATH, override=True)

REDSHIFT_CONFIG = {
    "host": os.environ["REDSHIFT_HOST"],
    "port": int(os.environ.get("REDSHIFT_PORT", 5439)),
    "database": os.environ["REDSHIFT_DATABASE"],
    "user": os.environ["REDSHIFT_USER"],
    "password": os.environ["REDSHIFT_PASSWORD"],
}

_missing = [k for k, v in REDSHIFT_CONFIG.items() if v in (None, "")]
if _missing:
    raise ValueError(f"Empty value(s) in .env: {_missing}")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Echo everything EXCEPT the password.
print({k: v for k, v in REDSHIFT_CONFIG.items() if k != "password"})
print(f"password: <set, {len(REDSHIFT_CONFIG['password'])} chars>")
print(f"extracts -> {RAW_DIR}")

## 3. Connection helpers

Rachel's snippet opens one long-lived `conn` and hands it to `pd.read_sql`. That works, but a connection left open across a long notebook session goes stale and the next cell fails with an opaque socket error. These helpers open a connection per query and always close it.

`cursor.fetch_dataframe()` is `redshift_connector`'s native DataFrame path — it avoids the pandas `UserWarning` you get from passing a raw DBAPI connection to `pd.read_sql`.

**Read-only discipline:** `run_query` blocks anything that isn't a `SELECT` / `WITH`. Use `execute_statement` deliberately if you ever need to write.

In [17]:
@contextmanager
def redshift_connection(**overrides):
    """Open a Redshift connection, guaranteed closed on exit."""
    conn = redshift_connector.connect(**{**REDSHIFT_CONFIG, **overrides})
    try:
        yield conn
    finally:
        conn.close()


_READ_ONLY_PREFIXES = ("select", "with", "show", "explain")


def run_query(sql: str, params=None) -> pd.DataFrame:
    """Run a read-only query and return a DataFrame.

    Parameters
    ----------
    sql : str
        A SELECT / WITH / SHOW / EXPLAIN statement.
    params : sequence, optional
        Values for %s placeholders. Use these instead of f-strings — string
        interpolation into SQL is how you end up with an injection bug or a
        silently wrong filter.
    """
    stripped = sql.strip().lstrip("(").lower()
    if not stripped.startswith(_READ_ONLY_PREFIXES):
        raise ValueError(
            "run_query is read-only. Use execute_statement() for DDL/DML."
        )

    with redshift_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            df = cur.fetch_dataframe()

    # fetch_dataframe returns None for a zero-row result set.
    return df if df is not None else pd.DataFrame()


def execute_statement(sql: str, params=None) -> None:
    """Run a statement that returns no rows (DDL, ALTER USER, ...). Commits."""
    with redshift_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params)
        conn.commit()


print("helpers defined: redshift_connection, run_query, execute_statement")

helpers defined: redshift_connection, run_query, execute_statement


## 4. Smoke test

First thing to run. If this returns a row, the connection is good.

| Symptom | Likely cause |
| --- | --- |
| Hangs, then timeout / "could not translate host name" | Not on VPN |
| `password authentication failed` | Wrong `REDSHIFT_PASSWORD` in `.env` (or you rotated it and didn't update `.env`) |
| `permission denied for relation ...` | Connection is fine; you need a GRANT — ask in `#ask-dataeng` |

In [18]:
run_query(
    """
    select
        current_user   as connected_as
      , current_database() as db
      , getdate()      as server_time_utc
      , version()      as redshift_version
    """
)

,connected_as,db,server_time_utc,redshift_version
0,swaminathan_chellappa,bw_analytics,2026-09-02 07:46:39,"PostgreSQL 8.0.2 on i686-pc-linux-gnu, compile..."


## 5. Project tables

The three tables the Proactive Churn model is built on, all in `analytics_mart_customer`:

| Table | Grain | Role |
| --- | --- | --- |
| `mart_school_customers` | one row per school (current state) | school-level attributes |
| `rpt_monthly_school_usage_trends` | one row per school × month | features — trailing averages, deltas, product score |
| `rpt_school_churn_cases` | one row per SaaS cancellation case | labels |


In [27]:
SCHEMA = "analytics_mart_customer"

TABLES = {
    "customers":      f"{SCHEMA}.mart_school_customers",
    "usage_trends":   f"{SCHEMA}.rpt_monthly_school_usage_trends",
    "churn_cases":    f"{SCHEMA}.rpt_school_churn_cases"
}

TABLES

{'customers': 'analytics_mart_customer.mart_school_customers',
 'usage_trends': 'analytics_mart_customer.rpt_monthly_school_usage_trends',
 'churn_cases': 'analytics_mart_customer.rpt_school_churn_cases'}

### 5a. Confirm access and row counts

Cheap query that tells you which tables you actually have `SELECT` on. If one errors with `permission denied`, note the table name and ask in `#ask-dataeng`.

In [28]:
for name, table in TABLES.items():
    try:
        n = run_query(f"select count(*) as n from {table}").iloc[0, 0]
        print(f"{name:<16} {table:<58} {n:>12,} rows")
    except Exception as e:
        print(f"{name:<16} {table:<58} FAILED: {type(e).__name__}: {e}")

customers        analytics_mart_customer.mart_school_customers                   370,839 rows
usage_trends     analytics_mart_customer.rpt_monthly_school_usage_trends       2,201,889 rows
churn_cases      analytics_mart_customer.rpt_school_churn_cases                  224,825 rows


### 5b. Inspect columns

Schema from the catalog — faster than `SELECT *` and it gives you the types, which you need for the feature spec.

In [29]:
def describe_table(table: str) -> pd.DataFrame:
    schema, name = table.split(".", 1)
    return run_query(
        """
        select
            ordinal_position as pos
          , column_name
          , data_type
          , is_nullable
        from information_schema.columns
        where table_schema = %s
          and table_name   = %s
        order by ordinal_position
        """,
        params=(schema, name),
    )


describe_table(TABLES["usage_trends"])

,pos,column_name,data_type,is_nullable


---

## 6. v1 dataset extract

Everything below is driven by the constants in the next cell. Change a constant, re-run, get a new dataset — no SQL editing.

### Decisions baked in

| Decision | Value | Source |
| --- | --- | --- |
| Observation window | 2024-01 → 2025-12 | train 2024 / test 2025 |
| Label horizon | next 3 months | spec |
| Label event | `is_reportable_request` | spec ("future requester") |
| Target-population gate | **not applied** | Rachel |
| Embargo | last 3 months of train year | spec (boundary leak) |
| Feature count | 10 (narrow first cut) | your call |
| `product_score` | baseline, not a feature | spec |

### Why `is_reportable_request` for the label

The spec framed positives as *requesters* — "each future requester into up to three positive observation-months." `is_reportable_request` is the warehouse's own definition of that event: the last eligible request case per school per case-created month, deduped to one row per school-month, coalesced to false so it is never null.

This deliberately **counts saves as positives**. A school that asks to cancel and gets talked out of it is exactly the case a proactive model exists to catch. Using `is_reportable_churn` instead would make every successful save a negative and teach the model to ignore the saveable cases. Flip `LABEL_FLAG` if you want the churn framing.

Choosing the request flag also disposes of `duplicate_or_merged` and its siblings automatically — those are *churn* classifications, and the request flag never touches them.

### Two things I chose that you should confirm

1. **`POPULATION_FILTER = "is_premium_sub_or_invoice"`** — premium this month. The actionable base for retention outreach. The alternative, `has_saas_subscription_this_month`, is broader. Not a decision I found in the spec.
2. **Column type assumptions** — the query treats `dim_month` as a date and the flags as booleans. If a flag is stored as `0/1`, Redshift will reject a bare predicate; change the constant to `"is_premium_sub_or_invoice = 1"`. `POPULATION_FILTER` is raw SQL precisely so this is a one-word fix.

In [ ]:
OBS_START = "2024-01-01"          # inclusive
OBS_END   = "2026-01-01"          # exclusive -> train 2024 + test 2025
LABEL_HORIZON_MONTHS = 3          # forward window
EMBARGO_MONTHS       = 3          # tail of train year dropped (boundary leak)

TRAIN_YEAR = 2024
TEST_YEAR  = 2025

LABEL_FLAG = "is_reportable_request"   # or "is_reportable_churn"
APPLY_TARGET_POPULATION = False        
POPULATION_FILTER = "is_premium_sub_or_invoice"   # raw SQL predicate

KEYS = ["school_id", "dim_month"]

FEATURES = [
    "customer_segment",                           # categorical
    "months_since_first_premium_sub_or_invoice",  # tenure
    "months_since_last_cancel_request",           # LEAK RISK
    "running_saas_cancel_request_count",          # LEAK RISK
    "months_since_last_failed_payment_case",
    "product_active_students",
    "delta_product_active_students_vs_t3m",
    "school_payer_attach_rate",
    "delta_school_payer_attach_rate_vs_t3m",
    "delta_billing_net_gpv_local_vs_t3m",
]

PASSTHROUGH = ["is_gov", "product_score"]  # context + baseline, NOT features

OBS_COLS = KEYS + FEATURES + PASSTHROUGH

_obs = "\n        , ".join(OBS_COLS)
_out = "\n        , ".join(f"o.{c}" for c in OBS_COLS)
_grp = ", ".join(str(i + 1) for i in range(len(OBS_COLS)))
_target = "      and is_target_population\n" if APPLY_TARGET_POPULATION else ""

EXTRACT_SQL = f"""
with obs as (
    select
          {_obs}
    from {TABLES['usage_trends']}
    where dim_month >= date '{OBS_START}'
      and dim_month <  date '{OBS_END}'
      and {POPULATION_FILTER}
)
, events as (
    select distinct
          school_id
        , date_trunc('month', case_create_or_churn_date)::date as event_month
    from {TABLES['churn_cases']}
    where {LABEL_FLAG}
{_target})
select
          {_out}
        , max(case when e.school_id is not null then 1 else 0 end) as label
from obs o
left join events e
       on e.school_id   = o.school_id
      and e.event_month >  o.dim_month
      and e.event_month <= dateadd(month, {LABEL_HORIZON_MONTHS}, o.dim_month)
group by {_grp}
"""

print(f"label      : {LABEL_FLAG}")
print(f"target pop : {'applied' if APPLY_TARGET_POPULATION else 'SKIPPED (Rachel)'}")
print(f"features   : {len(FEATURES)}")
print(EXTRACT_SQL)

In [ ]:
_t0 = time.time()
raw = run_query(EXTRACT_SQL)
print(f"{len(raw):,} rows x {raw.shape[1]} cols in {time.time() - _t0:.1f}s")
raw.head()

### 6a. What the label is actually made of

Before trusting the base rate, look at the event mix. This is a separate small query against the case table over the same window — it shows how the request flag and the churn classifications overlap, and how much the target-population gate would have removed had you applied it.

In [ ]:
EVENT_MIX_SQL = f"""
select
      coalesce(churn_classification, '(no churn event)') as churn_classification
    , count(*)                                            as cases
    , sum(case when is_reportable_request then 1 else 0 end) as reportable_requests
    , sum(case when is_reportable_churn   then 1 else 0 end) as reportable_churns
    , sum(case when is_target_population  then 1 else 0 end) as in_target_population
from {TABLES['churn_cases']}
where case_create_or_churn_date >= date '{OBS_START}'
  and case_create_or_churn_date <  date '{OBS_END}'
group by 1
order by cases desc
"""

run_query(EVENT_MIX_SQL)

### 6b. Split assignment and validation

The embargo is the point worth understanding. A December 2024 observation row has a label window reaching into March 2025 — so training on it leaks test-period information, and its near-identical January 2025 twin sits on the other side of the seam. Dropping the last `EMBARGO_MONTHS` of the train year closes both holes.

Rows are labelled rather than deleted, so you can see exactly what the embargo cost.

In [ ]:
df = raw.copy()
df["dim_month"] = pd.to_datetime(df["dim_month"])

_train_cutoff = pd.Timestamp(year=TRAIN_YEAR, month=12, day=1) - pd.DateOffset(
    months=EMBARGO_MONTHS
)

def _assign(m):
    if m.year == TRAIN_YEAR:
        return "train" if m <= _train_cutoff else "embargo"
    if m.year == TEST_YEAR:
        return "test"
    return "other"

df["split"] = df["dim_month"].map(_assign)

print(f"embargo drops train months after {_train_cutoff:%Y-%m}\n")
print(
    df.groupby("split")
      .agg(rows=("label", "size"), positives=("label", "sum"), rate=("label", "mean"))
      .assign(rate=lambda d: (d["rate"] * 100).round(2).astype(str) + "%")
)

_tr = df[df["split"] == "train"]
print(f"\ntrain rows      {len(_tr):>10,}    reference: 306,743 (looser pop here)")
print(f"train positives {int(_tr['label'].sum()):>10,}    reference:  10,149")
print(f"train pos rate  {_tr['label'].mean():>10.2%}    reference:   3.31%")
print(f"train schools   {_tr['school_id'].nunique():>10,}    reference:  31,633")
print(f"\ngov schools included: {int(df['is_gov'].sum()):,} rows"
      f"  (is_target_population would have excluded these)")
print("\nnulls per column:")
print(df.isna().sum().loc[lambda s: s > 0].sort_values(ascending=False))

In [ ]:
X_COLS = FEATURES          # product_score and is_gov deliberately excluded
Y_COL  = "label"

out = df.copy()
for c in out.select_dtypes(include=["float64"]).columns:
    out[c] = out[c].astype("float32")
out["label"] = out["label"].astype("int8")
for c in ["customer_segment", "split"]:
    out[c] = out[c].astype("category")

OUT = RAW_DIR / f"churn_v1_{LABEL_FLAG}_{TRAIN_YEAR}_{TEST_YEAR}.parquet"
out.to_parquet(OUT, index=False)

print(f"saved      {OUT.name}")
print(f"on disk    {OUT.stat().st_size / 1e6:.1f} MB")
print(f"in memory  {out.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nX_COLS ({len(X_COLS)}): {X_COLS}")
print(f"Y_COL: {Y_COL}")